# 01 - Clinical Data Ingestion, De-Identification & UMLS Entity Extraction

**Data Analytics Capstone - Research Pipeline**  
**Author:** Suddhasatwa Bhaumik  
**Institution:** Walsh College (QM640)  

---

### Overview
This notebook implements the data preprocessing stage of the **Clinical Summarization KG-RAG** framework:
1. **Data Ingestion**: Loads unstructured clinical discharge summaries from the MIMIC-IV dataset.
2. **Data Cleaning & PHI Masking**: Normalizes text formatting and strips out de-identification brackets (e.g. `[** Name **]`).
3. **Biomedical Named Entity Recognition (NER)**: Extracts clinical entities (symptoms, diagnoses, treatments) and maps them to canonical **UMLS Concept Unique Identifiers (CUIs)** using `scispacy`.
4. **Exploratory Data Analysis (EDA)**: Analyzes note token distributions and entity frequencies, logging summary metrics to BigQuery.


In [ ]:
import os
import re
import json
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Ensure working directory is the repository root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Current Working Directory: {os.getcwd()}")


## 1. Load Data & Schema Inspection
We load the raw clinical discharge notes from `data/raw/discharge.csv`. If the dataset does not exist locally, we create a representative sample based on the MIMIC-IV schema.


In [ ]:
from src.data_processor import DataProcessor

raw_data_path = "data/raw/discharge.csv"
processed_data_path = "data/processed/processed_data.csv"

# Ensure directories exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Generate or load raw data
if not os.path.exists(raw_data_path):
    print("Creating sample MIMIC-IV discharge summaries dataset...")
    sample_notes = {
        "hadm_id": [10001, 10002, 10003, 10004, 10005],
        "text": [
            "Patient [** Name **] is a 65yo male presenting with shortness of breath. Diagnosed with acute Asthma. Administered Albuterol inhaler. Status improved.",
            "Patient presented with severe chest pain. Diagnosed with Myocardial Infarction. Prescribed Aspirin and Metoprolol. Referred to cardiology.",
            "Subject presented with high fever and cough. Clinical workup confirmed Bacterial Pneumonia. Started on Amoxicillin therapy.",
            "Patient admitted for acute severe abdominal pain. Workup confirmed Acute Appendicitis. Administered IV Antibiotics and scheduled for appendectomy.",
            "The patient is a 72yo female admitted with dizziness and confusion. Diagnosed with Dehydration and Electrolyte Imbalance. Given IV Fluids."
        ]
    }
    df_raw = pd.DataFrame(sample_notes)
    df_raw.to_csv(raw_data_path, index=False)
else:
    df_raw = pd.read_csv(raw_data_path)

print(f"Loaded raw dataset shape: {df_raw.shape}")
df_raw.head()


## 2. Text Cleaning and De-Identification
We apply regular expressions to remove de-identification placeholders, normalize whitespaces, and preserve clinical narrative integrity.


In [ ]:
processor = DataProcessor()

# Test cleaning on sample clinical string
sample_raw = "Patient [** First Name **] [** Last Name **] presented with   dyspnea.\n\n\nDiagnosed with [** Hospital 12 **] asthma."
sample_cleaned = processor.clean_text(sample_raw)

print("--- RAW TEXT ---")
print(sample_raw)
print("\n--- CLEANED TEXT ---")
print(sample_cleaned)


## 3. Biomedical Entity Extraction & UMLS Linking
Extract biomedical named entities (diseases, drugs, procedures, symptoms) and resolve them to UMLS Concept Unique Identifiers (CUIs).


In [ ]:
# Extract entities for sample note
sample_entities = processor.extract_entities(sample_cleaned)
print(f"Extracted {len(sample_entities)} UMLS concepts:")
for ent in sample_entities:
    print(f"  • {ent['text']} -> CUI: {ent['cui']} ({ent['name']}) | Semantic Types: {ent.get('types', [])}")


## 4. Full Dataset Preprocessing & Batch Export
We process all raw clinical notes, generate structured annotations, and export the processed dataset.


In [ ]:
processor.process_csv(
    input_filepath=raw_data_path,
    output_filepath=processed_data_path
)

df_processed = pd.read_csv(processed_data_path)
print(f"Processed dataset saved to {processed_data_path} with {len(df_processed)} rows.")
df_processed[['hadm_id', 'cleaned_text', 'entities']].head()


## 5. Exploratory Data Analysis (EDA)
Inspect distribution of note character/word lengths and top extracted UMLS concepts.


In [ ]:
# Compute token and character length distributions
df_processed['char_count'] = df_processed['cleaned_text'].apply(len)
df_processed['word_count'] = df_processed['cleaned_text'].apply(lambda x: len(str(x).split()))

print("--- Summary Statistics ---")
print(df_processed[['char_count', 'word_count']].describe())

# Parse top UMLS entities
all_cuis = []
for ent_json in df_processed['entities']:
    if pd.notna(ent_json):
        try:
            for ent in json.loads(ent_json):
                all_cuis.append(ent.get('name', ent.get('cui')))
        except Exception:
            pass

cui_series = pd.Series(all_cuis).value_counts().head(10)
print("\n--- Top 10 Extracted Clinical Concepts ---")
print(cui_series)
